## Adzuna API

Con este script me conecto a la API de Adzuna y me descargo ofertas de trabajo de España.

In [ ]:
import requests
import pandas as pd
import math
import time 

# Parametros de la API de Adzuna
APP_ID = "######"
APP_KEY = "##############################"

RESULTS_PER_PAGE = 50 # el máximo de resultados por petición que permite la API
COUNTRY = 'es' 

all_jobs_data = [] # Lista donde se irán almacenando los resultados de cada página en formato de diccionario

# URL base
base_url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/"
base_params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": RESULTS_PER_PAGE,
}

# Primera petición para obtener el número total de resultados y calcular el número de páginas necesarias
first_page_url = f"{base_url}1"
try:
    response = requests.get(first_page_url, params=base_params)
    response.raise_for_status() #si el servidor devuelve un error 4xx o 5xx se lanzará la excepción
    data = response.json()

    total_results = data.get('count', 0) #se extrae el valor del campo 'count' de la respuesta, que indica el número total de resultados globales

    if total_results == 0:
        print("No se encontraron resultados para los criterios de búsqueda.")
        exit()

    # Calcular el número total de páginas necesarias
    total_pages = math.ceil(total_results / RESULTS_PER_PAGE)

    print(f"Total de trabajos encontrados: {total_results}")
    print(f"Número total de páginas a consultar: {total_pages}")

    # Se añaden los resultados de la primera página a la lista principal
    all_jobs_data.extend(data.get('results', [])) # [] es el valor por defecto si no se encuentra 'results'

except requests.exceptions.RequestException as e:
    print(f"Error en la primera llamada a la API: {e}")
    total_pages = 0


# Bucle para recorrer las páginas restantes 
for page_num in range(2, total_pages + 1):
    page_url = f"{base_url}{page_num}"
    print(f"Página {page_num} de {total_pages}...")

    try:
        response = requests.get(page_url, params=base_params)
        response.raise_for_status()
        page_data = response.json()

        # Se añade el resultado de la página a la lista
        all_jobs_data.extend(page_data.get('results', []))

        # Una pausa entre cada petición para no sobrecargar la API
        time.sleep(0.5)

    except requests.exceptions.RequestException as e:
        print(f"Error al obtener la página {page_num}: {e}")
        continue 

# Crear el DataFrame-
df = pd.DataFrame(all_jobs_data)
df.to_csv("../data/raw/AdzunaAPI.csv", index=False, encoding='utf-8')

## Web Scrapping Glassdoor

In [1]:
import subprocess
import sys
import os
import pandas as pd
import time as _time
import random
from datetime import datetime

# Ruta al script de scraping (se ejecuta como subproceso para evitar conflicto de event loops con Jupyter)
SCRIPT_PATH = os.path.abspath("../src/scraping_functions.py")

In [ ]:
# Ciudades y puestos a buscar
CIUDADES = [
    "Madrid","Barcelona","Valencia","Sevilla","Zaragoza","Málaga",
    "Murcia","Bilbao","Alicante","Valladolid","Palma de Mallorca",
    "Las Palmas de Gran Canaria","San Sebastián","A Coruña","Granada",
    "Vigo","Córdoba","Santander","Pamplona","Oviedo","Gijón",
    "Vitoria","Santa Cruz de Tenerife","Salamanca","Burgos",
    "Cádiz","Tarragona","León","Jerez de la Frontera","Castellón de la Plana",
    "Albacete","Almería","Ávila","Badajoz","Cáceres","Ciudad Real",
    "Cuenca","Girona","Guadalajara","Huelva","Huesca",
    "Jaén","Logroño","Lleida","Lugo","Ourense","Palencia",
    "Pontevedra","Segovia","Soria","Teruel","Toledo","Zamora"
]

PUESTOS = [
    "Tecnología","Data","Software",
    "Marketing","Administración","Sanidad",          
    "Finanzas","Ingeniería","Ventas",           
    "Logística","Hostelería","Construcción",     
    "Educación","Recursos Humanos","Legal"            
]


# Una búsqueda por ciudad y por puesto
busquedas = []

# Fase 1: Recorrer cada ciudad (sin filtro de puesto)
for ciudad in CIUDADES:
    busquedas.append({
        'tipo': 'ciudad',
        'ciudad': ciudad,
        'puesto': '',
        'nombre': f"ciudad_{ciudad}".replace(" ", "-"),
    })

# Fase 2: Recorrer cada puesto (sin filtro de ciudad)
for puesto in PUESTOS:
    busquedas.append({
        'tipo': 'puesto',
        'ciudad': '',
        'puesto': puesto,
        'nombre': f"puesto_{puesto}".replace(" ", "-"),
    })

print(f"Total de búsquedas a realizar: {len(busquedas)}")
print(f"Ciudades: {len(CIUDADES)} búsquedas")
print(f"Puestos: {len(PUESTOS)} búsquedas")



# Ejecucion por lotes con guardado progresivo de cada csv

OUTPUT_DIR = os.path.abspath("../data/raw/glassdoor_parciales")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Detectar csv ya completados
ya_completados = set()
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".csv"):
        ya_completados.add(f.replace(".csv", ""))

pendientes = []
for b in busquedas:
    if b['nombre'] not in ya_completados:
        pendientes.append(b)

print(f"Ya completados: {len(ya_completados)}")
print(f"Pendientes: {len(pendientes)}")


for i, busqueda in enumerate(pendientes):
    csv_parcial = os.path.join(OUTPUT_DIR, f"{busqueda['nombre']}.csv")

    if busqueda['tipo'] == 'ciudad':
        label = f"[FASE 1 - Ciudad] {busqueda['ciudad']}"
        keyword_arg = ""
        location_arg = busqueda['ciudad']
        etiqueta_ciudad = busqueda['ciudad']
        etiqueta_puesto = "Todos"
    else:
        label = f"[FASE 2 - Puesto] {busqueda['puesto']}"
        keyword_arg = busqueda['puesto']
        location_arg = ""
        etiqueta_ciudad = "Todas"
        etiqueta_puesto = busqueda['puesto']

    print(f"\n[{i+1}/{len(pendientes)}] {label}")

#para cada busqueda pendiente ejecuta la función de webscrapping con los parámetros correspondientes
    try:
        result = subprocess.run(
            [sys.executable, SCRIPT_PATH, csv_parcial,
             etiqueta_ciudad, etiqueta_puesto, keyword_arg, location_arg],
            capture_output=True, text=True, encoding="utf-8", timeout=2600
        )

        if result.stdout:
            for line in result.stdout.strip().split('\n'):
                if 'Extraidas' in line or 'Guardados' in line or 'Sin resultados' in line or 'No se encontraron' in line:
                    print(f"  {line.strip()}")

        if result.returncode != 0 and result.stderr:
            print(f"Error: {result.stderr[:200]}")
    except subprocess.TimeoutExpired:
        print(f"Timeout (>40min)")

    # Pausa entre búsquedas para evitar ser bloqueado por glassdoor
    PAUSA_ENTRE_BUSQUEDAS = 60
    if i < len(pendientes) - 1:
        pausa = PAUSA_ENTRE_BUSQUEDAS + random.randint(0, 30) #añado un numero aleatorio para simular comportamiento humano
        print(f"Pausa de {pausa}s...")
        _time.sleep(pausa)


Total de búsquedas a realizar: 69
Ciudades: 53 búsquedas
Puestos: 16 búsquedas
Ya completados: 32
Pendientes: 37

[1/37] [FASE 1 - Ciudad] Albacete
  -> Extraidas 280 ofertas para ciudad=Albacete
  Guardados 280 resultados en g:\Mi unidad\UFV\TFM\TFM proyecto\data\raw\glassdoor_parciales\ciudad_Albacete.csv
  Pausa de 63s...

[2/37] [FASE 1 - Ciudad] Almería
  -> Extraidas 509 ofertas para ciudad=Almería
  Guardados 509 resultados en g:\Mi unidad\UFV\TFM\TFM proyecto\data\raw\glassdoor_parciales\ciudad_Almería.csv
  Pausa de 61s...

[3/37] [FASE 1 - Ciudad] Ávila
  -> Extraidas 128 ofertas para ciudad=Ávila
  Guardados 128 resultados en g:\Mi unidad\UFV\TFM\TFM proyecto\data\raw\glassdoor_parciales\ciudad_Ávila.csv
  Pausa de 61s...

[4/37] [FASE 1 - Ciudad] Badajoz
  -> Extraidas 399 ofertas para ciudad=Badajoz
  Guardados 399 resultados en g:\Mi unidad\UFV\TFM\TFM proyecto\data\raw\glassdoor_parciales\ciudad_Badajoz.csv
  Pausa de 72s...

[5/37] [FASE 1 - Ciudad] Cáceres
  -> Extraid

Este script utiliza varias funciones para hacer WebScraping sobre Glassdoor (con la libería playwright), se encarga de filtrar la búsqueda de empleo por varias ciudades. Esto ya que una sola búsqueda tiene un limite de unas 700 ofertas de trabajo, de esta forma se generan muchas búsquedas y se puede obtener mucha más información. Para cada ciudad y cada puesto se genera un csv, lo que permite ejecutar la función en varios momentos para ir descargando toda la información.

El scripts se encarga de ir seleccionando cada oferta, cerrar la pestaña de iniciar sesión, ir descargado todo el html y agrupandolo por las categorías relevantes (título, empresa, localización, salario, fecha, valoracion empresa, descripcion). Esto es gracias a playwright, que permite abrir y controlar el navegador de forma automática mediante el código.

A partir de la descripción más adelante se podran obtener el tipo de empleo (jornada completa, parcial...), la modalidad (presencial, remoto), el sector, el nivel de experiencia requerido, habilidades y beneficios.

Cada búsqueda ha tardado aproximadamente 30 minutos, por 68 búsquedas, se ha tardado 2040 minutos, unas 34 horas.